# BeverageDzAI — serveur GPU Qwen 7B/14B + reranker

Ce notebook démarre un serveur privé pour l'application locale BeverageDzAI : Qwen2.5 AWQ 4-bit pour la synthèse et BGE-reranker-v2-m3 pour le reranking. Il fonctionne dans Google Colab ou un notebook RunPod.

Dans Colab : **Runtime → Change runtime type → GPU**. Pour une démo fiable en 14B, préférer un GPU RunPod de 24 Go. Ne partagez jamais la clé affichée par ce notebook.

In [ ]:
MODEL_SIZE = "7B"  #@param ["7B", "14B"]
MAX_MODEL_LEN = 8192

import os, secrets, subprocess, sys, time, re, requests, torch
from pathlib import Path

if not torch.cuda.is_available():
    raise RuntimeError("GPU absent. Activez un runtime GPU avant de continuer.")
gpu_name = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if MODEL_SIZE == "14B" and gpu_gb < 14.5:
    print(f"VRAM {gpu_gb:.1f} Go insuffisante pour le profil 14B; passage automatique à 7B.")
    MODEL_SIZE = "7B"
MODEL_ID = f"Qwen/Qwen2.5-{MODEL_SIZE}-Instruct-AWQ"
RERANKER_DEVICE = "cpu" if MODEL_SIZE == "14B" or gpu_gb < 20 else "cuda"
GPU_UTIL = "0.92" if MODEL_SIZE == "14B" else ("0.72" if RERANKER_DEVICE == "cuda" else "0.88")
API_TOKEN = secrets.token_urlsafe(32)
print(f"GPU: {gpu_name} ({gpu_gb:.1f} Go)")
print(f"Générateur: {MODEL_ID}")
print(f"Reranker: BAAI/bge-reranker-v2-m3 sur {RERANKER_DEVICE}")

## 1. Installer les dépendances
Le téléchargement initial des modèles peut prendre plusieurs minutes.

In [ ]:
%pip uninstall -y torch torchvision torchaudio torchtext
%pip install -q vllm==0.29.0 sentence-transformers fastapi uvicorn httpx requests
%pip install -q --force-reinstall torch==2.13.0 torchvision==0.28.0 torchaudio==2.11.0
%pip install -q "setuptools>=77.0.3,<81" "numpy<2.5" jedi
print("Dépendances installées.")

## 2. Démarrer Qwen avec vLLM
Attendez le message `vLLM prêt`. En 14B, le premier téléchargement est volumineux.

In [ ]:
vllm_log = open("/tmp/beverage-vllm.log", "w")
vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_ID,
    "--served-model-name", "beverage-qwen",
    "--host", "127.0.0.1", "--port", "8001",
    "--quantization", "awq",
    "--dtype", "half",
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", GPU_UTIL,
    "--max-num-seqs", "1",
    "--enforce-eager",
]
vllm_process = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)
deadline = time.time() + 1800
while time.time() < deadline:
    if vllm_process.poll() is not None:
        print(Path("/tmp/beverage-vllm.log").read_text()[-6000:])
        raise RuntimeError("vLLM s'est arrêté. Essayez MODEL_SIZE='7B' ou un GPU avec plus de VRAM.")
    try:
        if requests.get("http://127.0.0.1:8001/v1/models", timeout=5).ok:
            print("vLLM prêt.")
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError("vLLM n'a pas démarré dans les 30 minutes.")

## 3. Démarrer la passerelle authentifiée et le reranker
La passerelle protège toutes les routes publiques avec une clé Bearer et expose `/v1/chat/completions` et `/rerank`.

In [ ]:
gateway_code = r'''
import os
import httpx
from fastapi import FastAPI, HTTPException, Request, Response
from pydantic import BaseModel
from sentence_transformers import CrossEncoder

TOKEN = os.environ["BEVERAGE_GPU_TOKEN"]
RERANKER_MODEL = os.environ.get("BEVERAGE_RERANKER_MODEL", "BAAI/bge-reranker-v2-m3")
DEVICE = os.environ.get("BEVERAGE_RERANKER_DEVICE", "cpu")
app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
reranker = CrossEncoder(RERANKER_MODEL, device=DEVICE)

@app.middleware("http")
async def authenticate(request: Request, call_next):
    if request.headers.get("authorization") != f"Bearer {TOKEN}":
        return Response("Unauthorized", status_code=401)
    return await call_next(request)

class RerankRequest(BaseModel):
    model: str
    query: str
    documents: list[str]
    top_n: int = 10

@app.get("/health")
def health():
    return {"status": "ok", "reranker": RERANKER_MODEL, "device": DEVICE}

@app.post("/rerank")
def rerank(payload: RerankRequest):
    if not payload.documents:
        return {"results": []}
    scores = reranker.predict([(payload.query, doc) for doc in payload.documents], show_progress_bar=False)
    order = sorted(range(len(scores)), key=lambda i: float(scores[i]), reverse=True)[:payload.top_n]
    return {"results": [{"index": i, "relevance_score": float(scores[i])} for i in order]}

@app.api_route("/v1/{path:path}", methods=["GET", "POST"])
async def proxy_vllm(path: str, request: Request):
    body = await request.body()
    async with httpx.AsyncClient(timeout=900) as client:
        upstream = await client.request(
            request.method, f"http://127.0.0.1:8001/v1/{path}",
            content=body, headers={"content-type": request.headers.get("content-type", "application/json")}
        )
    return Response(content=upstream.content, status_code=upstream.status_code, media_type=upstream.headers.get("content-type"))
'''
Path("/tmp/beverage_gateway.py").write_text(gateway_code)
gateway_env = os.environ.copy()
gateway_env["BEVERAGE_GPU_TOKEN"] = API_TOKEN
gateway_env["BEVERAGE_RERANKER_DEVICE"] = RERANKER_DEVICE
gateway_log = open("/tmp/beverage-gateway.log", "w")
gateway_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "beverage_gateway:app", "--app-dir", "/tmp", "--host", "127.0.0.1", "--port", "8000"],
    env=gateway_env, stdout=gateway_log, stderr=subprocess.STDOUT
)
headers = {"Authorization": f"Bearer {API_TOKEN}"}
deadline = time.time() + 900
while time.time() < deadline:
    if gateway_process.poll() is not None:
        print(Path("/tmp/beverage-gateway.log").read_text()[-6000:])
        raise RuntimeError("La passerelle s'est arrêtée.")
    try:
        if requests.get("http://127.0.0.1:8000/health", headers=headers, timeout=5).ok:
            print("Passerelle et reranker prêts.")
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError("La passerelle n'a pas démarré.")

## 4. Créer l'URL temporaire sécurisée
Cette URL change à chaque redémarrage. La clé est obligatoire. Gardez les deux privées.

In [ ]:
cloudflared = Path("/tmp/cloudflared")
if not cloudflared.exists():
    data = requests.get("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", timeout=120).content
    cloudflared.write_bytes(data)
    cloudflared.chmod(0o755)
tunnel_process = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
public_url = None
deadline = time.time() + 120
while time.time() < deadline:
    line = tunnel_process.stdout.readline()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError("URL Cloudflare introuvable. Relancez cette cellule.")
print("\nCOPIEZ CES DEUX VALEURS DANS UN ENDROIT PRIVÉ:")
print("GPU_BASE_URL=" + public_url + "/v1")
print("GPU_API_KEY=" + API_TOKEN)
connection_file = Path("beverage_gpu_connection.env")
connection_file.write_text(
    f"VLLM_BASE_URL={public_url}/v1\nRERANKER_BASE_URL={public_url}\n"
    f"VLLM_API_KEY={API_TOKEN}\nRERANKER_API_KEY={API_TOKEN}\n"
)
try:
    from google.colab import files
    files.download(str(connection_file))
    print("Fichier de connexion téléchargé. Ne le partagez pas.")
except ImportError:
    print("Fichier de connexion créé:", connection_file.resolve())

## 5. Vérifier les deux services

In [ ]:
models = requests.get(public_url + "/v1/models", headers=headers, timeout=60)
models.raise_for_status()
rerank_test = requests.post(
    public_url + "/rerank", headers=headers, timeout=180,
    json={"model": "BAAI/bge-reranker-v2-m3", "query": "citrus oxidation", "documents": ["citral degrades by oxidation", "sugar sweetness"], "top_n": 2}
)
rerank_test.raise_for_status()
print("SERVEUR GPU VALIDÉ")
print("Modèles vLLM:", [item["id"] for item in models.json()["data"]])
print("Reranker:", rerank_test.json())

## 6. Garder le serveur actif
Lancez cette cellule et laissez l'onglet ouvert pendant la démo. Envoyez ensuite à Codex les valeurs `GPU_BASE_URL` et `GPU_API_KEY`. Arrêtez la cellule et le runtime après la démo.

In [ ]:
print("Serveur actif. Interrompez cette cellule pour arrêter.")
while all(process.poll() is None for process in [vllm_process, gateway_process, tunnel_process]):
    time.sleep(30)
raise RuntimeError("Un service s'est arrêté; consultez les logs /tmp/beverage-*.log")